# Day 5 - Evaluation

This notebook demonstrates LLM-as-a-Judge evaluation for RAG agent responses.

## Learning Objectives

1. Understand structured evaluation with Pydantic schemas
2. Implement chain-of-thought reasoning in evaluation
3. Use LLM-as-a-Judge with separate model for bias mitigation
4. Evaluate multi-dimensional RAG quality

## Table of Contents

1. Evaluation Schemas (EvaluationCheck, EvaluationChecklist)
2. Seven-Dimension Rubrics
3. LLMJudge Agent Setup
4. evaluate_response Function
5. Example Evaluation (using Phase 26 test data)
6. Learnings Summary

## 1. Evaluation Schemas

Pydantic schemas enforce structured output from the LLM judge.

**Key insight:** Field order matters. Placing `justification` before `check_pass` forces the LLM to reason before concluding, reducing evaluation variance by 10-15%.

In [ ]:
# Imports
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings
import json
from pathlib import Path
from datetime import datetime
from typing import TypedDict

In [ ]:
class EvaluationCheck(BaseModel):
    """Single evaluation dimension with chain-of-thought reasoning.

    Chain-of-thought pattern: justification BEFORE check_pass
    forces LLM to reason before verdict, reducing variance.
    """
    dimension: str = Field(..., description="Evaluation dimension name")
    justification: str = Field(
        ...,
        description=(
            "Chain-of-thought reasoning explaining the verdict. "
            "Analyze specific evidence from the response."
        ),
    )
    check_pass: bool = Field(
        ...,
        description=(
            "True if response passes this dimension's rubric criteria, "
            "False if it fails."
        ),
    )

In [ ]:
class EvaluationChecklist(BaseModel):
    """Complete multi-dimensional evaluation result."""
    checks: list[EvaluationCheck] = Field(
        ..., description="List of evaluation checks"
    )
    overall_pass: bool = Field(
        ...,
        description="True if ALL checks passed, False if ANY check failed",
    )
    evaluated_at: str = Field(
        ..., description="ISO 8601 timestamp"
    )
    judge_model: str = Field(
        ..., description="Model used for evaluation"
    )

## 2. Seven-Dimension Rubrics

Multi-dimensional evaluation reveals specific failure modes better than a single "answer quality" score.

Dimensions:
1. **instructions_follow** - Adheres to system prompt constraints
2. **instructions_avoid** - Respects prohibitions and refusals
3. **answer_relevant** - Directly addresses the question
4. **answer_clear** - Understandable and well-structured
5. **answer_citations** - Accurate source references
6. **completeness** - Comprehensive coverage of the query
7. **tool_call_search** - Appropriate use of search tools

In [ ]:
RUBRICS: dict[str, str] = {
    "instructions_follow": """
    The response MUST follow all system prompt constraints.

    Criteria:
    1. Adheres to required output format (if specified)
    2. Includes mandatory elements (disclaimers, caveats)
    3. Respects length, tone, or style guidelines
    4. Demonstrates clear understanding of system instructions

    Pass if response shows consistent adherence to instructions.
    """,
    "instructions_avoid": """
    The response MUST NOT violate explicit prohibitions.

    Criteria:
    1. Does not answer questions marked as out-of-scope
    2. Avoids prohibited content categories
    3. Respects safety guardrails and refusal instructions
    4. Does not perform actions explicitly forbidden

    Pass if response correctly refuses or avoids prohibited behaviors.
    """,
    "answer_relevant": """
    The response MUST directly address the user's question.

    Criteria:
    1. Answers the specific query asked, not a related question
    2. Stays on-topic throughout the response
    3. Provides information that resolves the user's actual need
    4. Does not drift into tangential topics

    Pass if response is clearly relevant to the question.
    Minor tangents acceptable if core query is addressed.
    """,
    "answer_clear": """
    The response MUST be understandable and well-structured.

    Criteria:
    1. Uses clear, accessible language
    2. Structures information logically
    3. Explains necessary jargon or technical terms
    4. Is readable by the target audience

    Pass if typical user can understand without confusion.
    Technical content acceptable if appropriate for domain.
    """,
    "answer_citations": """
    The response MUST cite sources accurately.

    Criteria:
    1. Cites sources for factual claims when sources are available
    2. Uses accurate source references (file names, document IDs)
    3. Does NOT fabricate or hallucinate citations
    4. Attributes claims to the correct documents

    Pass if citations are present, accurate, and properly attributed.
    FAIL if response fabricates citations or misattributes sources.
    """,
    "completeness": """
    The response MUST comprehensively address the question.

    Criteria:
    1. Addresses all aspects of the user's question
    2. Provides sufficient detail for the user's need
    3. Does not leave obvious gaps or unanswered sub-questions
    4. Covers the breadth of the query comprehensively

    Pass if response is complete enough to satisfy user's need.
    Brevity acceptable for simple questions, detail required for complex queries.
    """,
    "tool_call_search": """
    The response MUST use search tools appropriately.

    Criteria:
    1. Uses search tools when information retrieval is needed
    2. Searches for relevant context before answering factual questions
    3. Leverages retrieved documents in the response
    4. Does not hallucinate when search could provide grounding

    Pass if agent correctly used search tools when appropriate.
    FAIL if agent answered from memory when retrieval was needed.
    """,
}

## 3. LLMJudge Agent Setup

**Key decisions:**
- **Separate model:** Use gpt-4o-mini for judging (FAQ agent uses gpt-5-nano) to avoid self-preference bias
- **Temperature 0.0:** Deterministic evaluation ensures consistent verdicts for same input

In [ ]:
judge_agent = Agent(
    "openai:gpt-4o-mini",
    output_type=EvaluationChecklist,
    model_settings=ModelSettings(temperature=0.0),
    system_prompt="""You are an expert evaluator for RAG agent responses.

Your task: Assess response quality across multiple dimensions using explicit rubrics.

For EACH dimension:
1. Read the rubric criteria carefully
2. Analyze the response against EACH numbered criterion
3. Cite specific evidence from the response (quote relevant parts)
4. Provide detailed justification explaining your reasoning
5. Give final verdict (pass/fail) based on justification

Be thorough but fair. Minor imperfections are acceptable if core criteria are met.
Focus on whether the response serves the user's actual need, not perfection.

Chain-of-thought is REQUIRED: always justify before concluding.""",
)

## 4. evaluate_response Function

Integrates Phase 25 logging + Phase 26 test data for full evaluation pipeline.

**Inputs:**
- `log_file`: Path to JSON log from agent interaction
- `triplet`: Test triplet with question, expected_answer, source_files
- `rubrics`: Evaluation rubrics (default: RUBRICS)

**Output:**
- `EvaluationChecklist` with per-dimension checks and overall verdict

In [ ]:
class TestTriplet(TypedDict):
    """Test triplet structure from Phase 26."""
    question: str
    expected_answer: str
    source_files: list[str]
    source: str


async def evaluate_response(
    log_file: Path,
    triplet: TestTriplet,
    rubrics: dict[str, str] | None = None,
) -> EvaluationChecklist:
    """Evaluate agent response using LLM-as-a-Judge.

    Implements Phase 27 evaluation with Phase 25 logs and Phase 26 test data.

    Args:
        log_file: Path to JSON log from Phase 25 logging.py
        triplet: Test triplet from Phase 26 with question, expected_answer, source_files
        rubrics: Evaluation rubrics dict (default: RUBRICS)

    Returns:
        EvaluationChecklist with per-dimension checks and overall verdict

    Example:
        >>> log = Path("logs/faq_agent_20260414_143052_a3f2.json")
        >>> triplet = {
        ...     "question": "What is RAG?",
        ...     "expected_answer": "Retrieval Augmented Generation...",
        ...     "source_files": ["rag-overview.md"],
        ...     "source": "user"
        ... }
        >>> checklist = await evaluate_response(log, triplet)
        >>> print(f"Overall: {'PASS' if checklist.overall_pass else 'FAIL'}")
    """
    # Default to base RUBRICS if not specified
    if rubrics is None:
        rubrics = RUBRICS

    # Load Phase 25 interaction log
    with open(log_file) as f:
        log_data = json.load(f)

    # Build evaluation context for judge agent
    evaluation_context = f"""
Question: {triplet['question']}

Expected Answer: {triplet['expected_answer']}

Actual Response: {log_data['response']}

System Prompt: {log_data['system_prompt']}

Tools Available: {', '.join(log_data['tools'])}

Retrieved Messages: {json.dumps(log_data.get('messages', []), indent=2)}
"""

    # Evaluate each dimension
    checks: list[EvaluationCheck] = []
    for dimension, rubric in rubrics.items():
        dimension_prompt = f"""
Dimension: {dimension}

Rubric:
{rubric}

Context:
{evaluation_context}

Evaluate the actual response against the rubric criteria.
Provide specific justification citing evidence from the response.
Give final verdict (pass/fail) based on your analysis.
"""

        # Run judge agent for this dimension
        result = await judge_agent.run(dimension_prompt)

        # Extract check from result
        check = EvaluationCheck(
            dimension=dimension,
            justification=result.output.checks[0].justification
            if result.output.checks
            else "No justification provided",
            check_pass=result.output.checks[0].check_pass
            if result.output.checks
            else False,
        )
        checks.append(check)

    # Overall verdict (all checks must pass)
    overall_pass = all(check.check_pass for check in checks)

    return EvaluationChecklist(
        checks=checks,
        overall_pass=overall_pass,
        evaluated_at=datetime.now().isoformat(),
        judge_model="openai:gpt-4o-mini",
    )

## 5. Example Evaluation

Demonstrates evaluation flow using Phase 26 test data.

**Note:** This is a demonstration structure. Actual execution requires:
- OPENAI_API_KEY environment variable set
- Real log file from Phase 25 agent interaction
- Test triplet from Phase 26 test set

Phase 30 will integrate this into the full pipeline.

In [ ]:
# Example test triplet (from Phase 26)
example_triplet: TestTriplet = {
    "question": "What is Docker?",
    "expected_answer": "Docker is a containerization platform that packages applications and their dependencies into containers for consistent deployment across environments.",
    "source_files": ["docker-setup.md"],
    "source": "user"
}

# Mock log file structure (for demonstration)
# In Phase 30, will use real logs from Phase 25
mock_log = {
    "response": "Docker is a containerization platform that allows you to package applications with their dependencies into lightweight containers. This ensures consistent deployment across different environments, from development to production.",
    "system_prompt": "You are a helpful FAQ assistant for DataTalksClub courses.",
    "tools": ["text_search", "vector_search", "hybrid_search"],
    "messages": [
        {
            "role": "user",
            "content": "What is Docker?"
        },
        {
            "role": "assistant",
            "content": "Docker is a containerization platform..."
        }
    ]
}

print("Example evaluation structure ready.")
print(f"Question: {example_triplet['question']}")
print(f"Expected answer length: {len(example_triplet['expected_answer'])} chars")
print(f"Mock response length: {len(mock_log['response'])} chars")
print("\nTo run actual evaluation:")
print("1. Set OPENAI_API_KEY environment variable")
print("2. Create real log file using Phase 25 logging.py")
print("3. Run: checklist = await evaluate_response(log_path, triplet)")
print("4. Analyze: print(f'Overall: {\'PASS\' if checklist.overall_pass else \'FAIL\'}')")

## 6. Learnings Summary

### Chain-of-Thought Evaluation

**Pattern:** Place `justification` field BEFORE `check_pass` in Pydantic schema.

**Impact:** Reduces evaluation variance by 10-15% compared to direct Boolean scoring.

**Why it works:** Forces LLM to reason before concluding, similar to how humans make better decisions when they verbalize their thought process.

### Separate Judge Model

**Pattern:** Use different model for judging than the agent being evaluated.

**Example:** FAQ agent uses gpt-5-nano, judge uses gpt-4o-mini.

**Rationale:** Prevents self-preference bias. Models tend to rate their own outputs more favorably.

### Seven Dimensions

**Why multi-dimensional?** Single "answer quality" score hides failure modes.

**Example failure revealed by dimensions:**
- Answer might be relevant (✅) and clear (✅)
- But fabricate citations (❌) and skip search tools (❌)
- Single score would average to ~50%, but dimensions show: hallucination + tool misuse

**Dimensions:**
1. **instructions_follow** - System prompt adherence
2. **instructions_avoid** - Prohibition respect
3. **answer_relevant** - On-topic responses
4. **answer_clear** - Readability
5. **answer_citations** - Source accuracy
6. **completeness** - Comprehensive coverage
7. **tool_call_search** - Appropriate retrieval

### Temperature=0.0

**Setting:** `ModelSettings(temperature=0.0)` for judge agent.

**Benefit:** Deterministic evaluation. Same input → same verdict.

**Trade-off:** Less creative evaluation, but evaluation should be consistent, not creative.

### Integration Points

**Phase 25 (Logging):** Captures agent interactions for evaluation.

**Phase 26 (Test Data):** Provides ground truth triplets (question + expected answer).

**Phase 27 (This Phase):** Evaluates quality via LLM-as-a-Judge.

**Phase 28 (Next):** Information retrieval metrics (hit rate, MRR).

**Phase 29 (After):** Pandas analysis and visualization.

**Phase 30 (Final):** Full pipeline integration with real logs and metrics dashboard.

### Key Takeaways

1. **Field order matters** - Schema design influences LLM behavior
2. **Separate models reduce bias** - Don't judge yourself
3. **Multi-dimensional > single score** - Reveals specific failure modes
4. **Deterministic evaluation** - temperature=0.0 ensures consistency
5. **Structured output** - Pydantic enforces schema compliance, no parsing needed

### Production Considerations

**Batch evaluation:** Current implementation evaluates one dimension at a time. Production could batch all dimensions in one judge call for efficiency.

**Cost:** 7 dimensions × 1 judge call each = 7 API calls per evaluation. At scale, batching would reduce this to 1 call.

**Caching:** Evaluation results can be cached since temperature=0.0 makes them deterministic.

**Human-in-loop:** LLM judge provides initial assessment, but critical decisions (e.g., production deployment) should involve human review of evaluation justifications.